Stress Test for AI-Generated Hotel Review Detector

This notebook evaluates the robustness of the stylometry-based detector by testing it on manually constructed edge-case reviews.

While the model achieves near-perfect performance on the test set, we investigate whether this performance generalizes across different writing styles.

We focus on whether the detector truly identifies AI-generated text or simply relies on stylistic patterns such as fluency and structure.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
stress_categories = {
    "short_human": "Very short, informal human-like reviews",
    "messy_human": "Lowercase, unstructured, messy human writing",
    "polished_human": "Well-written human reviews with fluent language",
    "neutral": "Balanced and ambiguous tone",
    "ai_like": "Typical AI-style fluent and structured reviews"
}

In [5]:
from src.final_detector import FinalReviewDetector
from src.stylometry_features import extract_stylometry_features

detector = FinalReviewDetector()

def detect_hotel_review(text):
    features = extract_stylometry_features(text)
    result = detector.detect_review(
        review_text=text,
        extracted_features=features,
        use_calibration=True
    )
    return {
        "ai_probability": result.ai_probability,
        "ai_likeness_score": result.ai_likeness_score,
        "uncertainty_band": result.uncertainty_band,
        "predicted_label": result.predicted_label,
        "top_features": result.top_features
    }

In [3]:
test_cases = [
    {"type": "short_human", "text": "Room was ok. Good location, but noisy at night."},

    {"type": "messy_human", "text": "stayed 2 nights room kinda small but staff nice location good"},

    {"type": "polished_human", "text": "The room was clean and quiet, but the bathroom felt outdated and the breakfast was only average."},

    {"type": "neutral", "text": "The hotel was fine. Clean room, decent service, good location. Nothing special but nothing bad either."},

    {"type": "ai_like", "text": "This hotel exceeded expectations with excellent service, well-designed amenities, and a seamless guest experience."}
]

In [6]:
results = []

for case in test_cases:
    output = detect_hotel_review(case["text"])
    
    results.append({
        "stress_type": case["type"],
        "review_text": case["text"],
        "ai_probability": output["ai_probability"],
        "score": output["ai_likeness_score"],
        "band": output["uncertainty_band"],
        "predicted_label": output["predicted_label"],
        "top_features": output["top_features"]
    })

import pandas as pd
results_df = pd.DataFrame(results)
results_df

,stress_type,review_text,ai_probability,score,band,predicted_label,top_features
0,short_human,"Room was ok. Good location, but noisy at night.",0.9999,100,likely AI-generated,AI,"{'capital_letter_ratio': 0.0426, 'stopword_rat..."
1,messy_human,stayed 2 nights room kinda small but staff nic...,0.0000,0,likely human-written,Human,"{'capital_letter_ratio': 0.0, 'stopword_ratio'..."
2,polished_human,"The room was clean and quiet, but the bathroom...",1.0000,100,likely AI-generated,AI,"{'capital_letter_ratio': 0.0104, 'stopword_rat..."
3,neutral,"The hotel was fine. Clean room, decent service...",0.4237,42,uncertain,Human,"{'capital_letter_ratio': 0.0294, 'stopword_rat..."
4,ai_like,This hotel exceeded expectations with excellen...,0.9235,92,likely AI-generated,AI,"{'capital_letter_ratio': 0.0088, 'stopword_rat..."
